# 03 — Model Training (original — for reference)

> Loads pre-computed features from `results/cmv_comments_df.csv.zip`.
> This is the **original** training code kept as a reference baseline.
> For the improved evaluation see `04_honest_evaluation.ipynb`.


In [ ]:
import pandas as pd, zipfile

with zipfile.ZipFile('../results/cmv_comments_df.csv.zip') as z:
    comments_df = pd.read_csv(z.open(z.namelist()[0]))

print(comments_df.shape)
print(comments_df.columns.tolist())


# ８. Training a Model to Predict Persuasiveness

The final step would be to use all the features we created as input to multiple ML models to predict whether a comment would be convincing. We will train multiple models and search for the best model.

In [ ]:
comments_df.columns

In [ ]:
categorical_features = ["tone_google", 'tone_google_op']
binary_features = ["style_label", 'style_label_op']
numeric_features = ["sentiment", "adj_adv_ratio", 'fk_grade_level', 'fk_reading_ease', 'evidence_count',
                    'sentiment_op', 'adj_adv_ratio_op', 'fk_grade_level_op', 'fk_reading_ease_op', 'evidence_count_op']
label = "is_convincing"

## 8.1 Encode Categorical Features

In [ ]:
comments_df['is_formal'] = comments_df['style_label'].apply(lambda x: 1 if x == 'formal' else 0)
comments_df['is_formal_op'] = comments_df['style_label_op'].apply(lambda x: 1 if x == 'formal' else 0)

In [ ]:
comments_df[['style_label', 'is_formal', 'style_label_op', 'is_formal_op']].head()


In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')
tone_encoded = onehot_encoder.fit_transform(comments_df[categorical_features])
tone_df = pd.DataFrame(tone_encoded, columns=onehot_encoder.get_feature_names_out())

comments_df = comments_df.drop(columns=categorical_features).reset_index(drop=True)
comments_df = pd.concat([comments_df, tone_df], axis=1)

In [ ]:
tone_df.columns


## 8.2 Split to Train and Test

First, we want to create a final dataframe with only the relevant features

In [ ]:
features = tone_df.columns.tolist() + ['is_formal', 'is_formal_op'] + numeric_features + [label]
data = comments_df[features]
data.head()

In [ ]:
data.columns

In [ ]:
import plotly.express as px

labels=["Convincing","Not Convincing"]

convincing = data["is_convincing"].value_counts().tolist()
values = [convincing[0], convincing[1]]

fig = px.pie(values=data["is_convincing"].value_counts(), names=labels , width=700, height=400, color_discrete_sequence=['skyblue', 'orange']
             ,title="Convincing Vs. Not-Convincing Comments")
fig.show()

In [ ]:
data["is_convincing"].value_counts()

In [ ]:
# Sepratating & assigning features and target columns to X & y
y = data["is_convincing"]
X = data.drop('is_convincing',axis=1)
X.shape, y.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,stratify=y,test_size = 0.2, random_state = 42)

X_train.shape, X_test.shape

In [ ]:
import pandas as pd
pd.concat([X_train, y_train], axis=1).to_csv('../results/combined_train_data.csv', index=False)
pd.concat([X_test,  y_test],  axis=1).to_csv('../results/combined_test_data.csv',  index=False)
print('Saved train/test splits')


## 8.3 Scale Data

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled_features = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_features = scaler.transform(X_test[numeric_features])

X_train[numeric_features] = X_train_scaled_features
X_test[numeric_features] = X_test_scaled_features

## 8.4 Train and Evaluate Models - Version 1

Important: The following training of the models was executed on a previous version of the data, that did not include the feature 'use_of_presuasive_lang' and the embeddings.

In a later phase we added the additional features. Below (section 8.5) we include re-training of the models with the new features.

In [ ]:
pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display, Markdown
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

In [ ]:
import random
np.random.seed(402) # we would like to set the random seed, to be able to reproduce the results, because we use Random Forest model.
scoring='f1'

models = [
    ('logistic regression', GridSearchCV(
        LogisticRegression(random_state=42, class_weight='balanced'),
         {'C': [0.1, 1, 10, 100],
          'penalty': ['l1', 'l2'],
          'solver': ['lbfgs', 'newton-cholesky', 'liblinear']},
        cv=5, scoring=scoring)),

    ('SVC', GridSearchCV(
        SVC(random_state=42, class_weight='balanced'),
         {'C': [0.1, 1, 10],
          'kernel': ['linear', 'rbf'],
          'gamma': [0.1, 1, 10],
          'degree': [2, 3, 4]},
        cv= 5, scoring=scoring)),

    ('decision tree', GridSearchCV(
        DecisionTreeClassifier(random_state=42, class_weight='balanced'),
         {'max_depth': [None, 5, 10, 15],
          'min_samples_split': [2, 5, 10],
          'min_samples_leaf': [1, 5, 20],
          'max_features': [None, 'sqrt', 'log2']},
        cv= 5, scoring=scoring)),

    ('random forest', GridSearchCV(
        RandomForestClassifier(random_state=42, class_weight='balanced'),
         {'n_estimators': [10, 100, 1000],
          'max_depth': [None, 5, 10, 15],
          'min_samples_split': [2, 5, 10],
          'min_samples_leaf': [1, 5, 20]},
        cv=5, scoring=scoring)),

    ('xgboost', GridSearchCV(
        xgb.XGBClassifier(random_state=42, scale_pos_weight=1),  # scale_pos_weight can be used for imbalance
        {'n_estimators': [50, 100, 200],
         'learning_rate': [0.01, 0.1, 0.3],
         'max_depth': [3, 6, 9],
         'gamma': [0, 0.1, 0.2]},
        cv=5, scoring=scoring))
    ]

In [ ]:
threshold = 0.5 # Convert probabilities to binary predictions (0 or 1)

def measure_error(y_true, y_pred_proba, label):
    y_pred = (y_pred_proba >= threshold).astype(int)
    return pd.Series({"accuracy": accuracy_score(y_true, y_pred),
                      'recall': recall_score(y_true, y_pred, average ='macro'),
                      'f1': f1_score(y_true, y_pred, average ='macro')},
                      name=label)

def execute_models(X_train, y_train, X_test, y_test, models):
  metrics = list()
  for model_name, model in models:
      print(f"Training {model_name}...")

      model.fit(X_train, y_train)
      if hasattr(model, "predict_proba"):
        y_test_positive_pred = model.predict_proba(X_test)[:,1]
        y_train_positive_pred = model.predict_proba(X_train)[:,1]

      elif hasattr(model, "decision_function"):  # Use decision_function for SVC
            y_test_positive_pred = model.decision_function(X_test)
            y_train_positive_pred = model.decision_function(X_train)
      else:
            raise ValueError(f"Model {model_name} does not support probability prediction.")

      # Evaluate
      print(f"Evaluating {model_name} ...")
      train_test_full_error = pd.concat([measure_error(y_test, y_test_positive_pred, 'test '+ model_name),
                                        measure_error(y_train, y_train_positive_pred, 'train '+ model_name)],
                                        axis=1)
      metrics.append(train_test_full_error)

  metrics = pd.concat(metrics, axis=1)
  return metrics

In [ ]:
metrics = execute_models(X_train, y_train, X_test, y_test, models)

In [ ]:
display(Markdown('**ML models**'))
display(metrics)

### Initial training results
* Best Accuracy: XGBoost - 0.870416
* Best recall: Random Forest - 0.673500
* Best F1: Random Forest - 0.653320

The results seem not good enough, so we will try to use undersampling to solve the imbalanced data issue that may affect the models ability to predict correctly.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

undersampler = RandomUnderSampler(
    sampling_strategy='auto',
    random_state=42
)
X_train_resampled, y_train_resampled = undersampler.fit_resample(X_train, y_train)

In [ ]:
class_distribution = pd.Series(y_train_resampled).value_counts()

# Labels and sizes for the pie chart
labels = ['Non Convincing', 'Convincing']
sizes = class_distribution.values

# Plot the pie chart
plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=['skyblue', 'orange'])
plt.title('Distribution of Convincing and Non Convincing Comments (Resampled)')
plt.show()

In [ ]:
metrics = execute_models(X_train_resampled, y_train_resampled, X_test, y_test, models)

In [ ]:
display(Markdown('**ML models - Balanced Data**'))
display(metrics)

### Training with Undersampling Results
* Best Accuracy: SVC - 0.694377
* Best recall: Random Forest - 0.698875
* Best F1: XGBoost - 0.584065

### Previous Results
* Best Accuracy: XGBoost - 0.870416
* Best recall: Random Forest - 0.673500
* Best F1: Random Forest - 0.653320

### Best Results So Far
* Best Accuracy: XGBoost (initial training) - 0.870416
* Best recall: Random Forest (Undersampling) - 0.698875
* Best F1: Random Forest (initial training) - 0.653320

We will try to balance the dataset with Oversampling instead - it means to create synthetic data for the minority class (convincing comments).

In [ ]:
from imblearn.over_sampling import SMOTE

oversample_strategy = 'auto'

smote = SMOTE(sampling_strategy=oversample_strategy, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

Make sure that we have the same amount of comments in both classes:

In [ ]:
y_train_resampled.value_counts()

Finally let's re-run the models

In [ ]:
metrics = execute_models(X_train_resampled, y_train_resampled, X_test, y_test, models)

In [ ]:
display(Markdown('**ML models - After Oversampling**'))
display(metrics)

### Training with Oversampling Results
* Best Accuracy: SVC - 0.852078
* Best recall: Random Forest - 0.500545
* Best F1: XGBoost - 0.472555

### Previous Results
* Best Accuracy: XGBoost (initial training) - 0.870416
* Best recall: Random Forest (Undersampling) - 0.698875
* Best F1: Random Forest (initial training) - 0.653320

### Best Results So Far
* Best Accuracy: XGBoost (initial training) - 0.870416
* Best recall: Random Forest (Undersampling) - 0.698875
* Best F1: Random Forest (initial training) - 0.653320


Oversampling did not improve any of the metrics.

## 8.5 Train and Evaluate Models - Version 2

Due to low performance, we added to our dataset new features. The first is the 'use_of_persuasive_lang' and the second is actually a set of features, which are the embedding features, see sections 6.8 and 6.9 above for documentation.

We will re-define the columns and execute all pre-processing like done for the first version of training.

In [ ]:
chunk_size = 10
cols = list(comments_df.columns)
for i in range(0, len(cols), chunk_size):
    print(str(cols[i:i + chunk_size]).strip('[]'))

In [ ]:
categorical_features = ["tone_google", 'tone_google_op']
binary_features = ["style_label", 'style_label_op', 'use_of_persuasive_lang']
numeric_features = ["sentiment", "adj_adv_ratio", 'fk_grade_level', 'fk_reading_ease', 'evidence_count',
                    'sentiment_op', 'adj_adv_ratio_op', 'fk_grade_level_op', 'fk_reading_ease_op', 'evidence_count_op']
embedding_columns = [f'embedding_{i}' for i in range(143)]
label = "is_convincing"

Encode the categorical features:

In [ ]:
comments_df['is_formal'] = comments_df['style_label'].apply(lambda x: 1 if x == 'formal' else 0)
comments_df['is_formal_op'] = comments_df['style_label_op'].apply(lambda x: 1 if x == 'formal' else 0)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False, drop='first')
tone_encoded = onehot_encoder.fit_transform(comments_df[categorical_features])
tone_df = pd.DataFrame(tone_encoded, columns=onehot_encoder.get_feature_names_out())

comments_df = comments_df.drop(columns=categorical_features).reset_index(drop=True)
comments_df = pd.concat([comments_df, tone_df], axis=1)

Split to train and test:

In [ ]:
features = tone_df.columns.tolist() + ['is_formal', 'is_formal_op', 'use_of_persuasive_lang'] + numeric_features + embedding_columns + [label]
data = comments_df[features]

y = data["is_convincing"]
X = data.drop('is_convincing',axis=1)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,stratify=y,test_size = 0.2, random_state = 42)

In [ ]:
import pandas as pd
pd.concat([X_train, y_train], axis=1).to_csv('../results/combined_train_data_v2.csv', index=False)
pd.concat([X_test,  y_test],  axis=1).to_csv('../results/combined_test_data_v2.csv',  index=False)
print('Saved train/test splits v2')


Scale the numeric features:

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_scaled_features = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_features = scaler.transform(X_test[numeric_features])

X_train[numeric_features] = X_train_scaled_features
X_test[numeric_features] = X_test_scaled_features

### Feature Selection
Including the embedding in the features list dramatically increases the number of features, so we will apply feature selection using the RFE - Recursive Feature Elimination technique.


We will try with 100 features.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Feature selection using RFE (Recursive Feature Elimination)
number_of_features = 100
selector = RFE(estimator=RandomForestClassifier(), n_features_to_select=number_of_features)
X_train_selected = selector.fit_transform(X_train, y_train)

In [ ]:
selector.get_feature_names_out()

In [ ]:
print(list(selector.get_feature_names_out()))

In [ ]:
X_train_selected.shape

Finally we can train the models.

In [ ]:
X_test_selected = selector.transform(X_test)

In [ ]:
metrics = execute_models(X_train_selected, y_train, X_test_selected, y_test, models)

In [ ]:
display(Markdown('**ML models - Selected Features**'))
display(metrics)

### Training with Embedding + Feature Selection Results
* Best Accuracy: XGBoost - 0.869193
* Best recall: Logistic Regression - 0.686792
* Best F1: XGBoost - 0.643221

### Previous Results
* Best Accuracy: XGBoost (initial training) - 0.870416
* Best recall: Random Forest (Undersampling) - 0.698875
* Best F1: Random Forest (initial training) - 0.653320


### Best Results
* Best Accuracy: XGBoost (initial training) - 0.870416
* Best recall: Random Forest (Undersampling) - 0.698875
* Best F1: Random Forest (initial training) - 0.653320


As we see, adding the embedding and using feture selection did not improve the results.

# ９. Conclusion and Future Research

In this work we investigated comments from the CMV forum.


We created a new and rich dataset with comments, context (thread text by the author of the comment), original post and several features extracted from the comment + context text and from the original post text.


This dataset can be extended in the future by adding more features and can be used for furher studies.


Possible features to add (we didn't add them due to time limitations):
* Lexical Complexity:
  * Type-token ratio (TTR) – Measures vocabulary richness.
  * Word frequency analysis – Uses external corpora to determine whether a text contains rare or common words.
* Syntactic Complexity:
  * Parse tree depth – Longer, more nested sentences are syntactically complex.
  * Mean length of sentences (MLS) – Longer sentences generally indicate higher complexity.


The dataset can be downloded from [here](https://drive.google.com/file/d/1cJWCp6StLol6M9yQHdI6irssBkTi-dKO/view?usp=sharing).

Once we had the dataset, we trained multiple ML models and checked their ability to predict persuasiveness.

We saw that although participants in the CMV forum post their opinions with the intention of hearing opposing views and being open to persuasion, in practice, there is a relatively small number of messages that are marked as persuasive by those same writers. This leads to an imbalanced dataset, which can be a challenge for ML models. We tried 2 resampling methods to overcome this problem.

We also tried to add embedding of the original text and include this in the features. Since this adds a lot of features, we combined this change with Feature Selection technique. We chose to use 100 features, but of course, future researches can try different numbers of features to get improved results.

The best results we got from this research were:
* Best Accuracy: XGBoost (without embedding and without resampling) - 0.870416
* Best recall: Random Forest (without embedding, with undersampling) - 0.698875
* Best F1: Random Forest (without embedding and without resampling) - 0.653320

# １0. References

1. Priniski, J.H. & Horne, Z. (2018). [Attitude Change on Reddit’s Change My View](https://jpriniski.github.io/papers/cogsci-reddit.pdf). In T.T. Rogers, M. Rau, X. Zhu, & C. W. Kalish (Eds.), Proceedings of the 40th Annual Conference of the Cognitive Science Society (pp. 2276-2281). Austin, TX: Cognitive Science Society.

2. [GitHub repo](https://github.com/jpriniski/CMV) of source [1]

3. Xiao, L., Mensah, H. (2022). [How Does the Thread Level of a Comment Affect its Perceived Persuasiveness? A Reddit Study](https://doi.org/10.1007/978-3-031-10464-0_55). In: Arai, K. (eds) Intelligent Computing. SAI 2022. Lecture Notes in Networks and Systems, vol 507. Springer, Cham.

4. Ivan Habernal and Iryna Gurevych. 2016. [What makes a convincing argument? Empirical analysis and detecting attributes of convincingness in Web argumentation](https://aclanthology.org/D16-1129.pdf). In Proceedings of the 2016 Conference on Empirical Methods in Natural Language Processing, pages 1214–1223, Austin, Texas. Association for Computational Linguistics.



## This Work Repo
https://github.com/jct-nlp/change-my-view-2025
